# Training

---

### Libraries

In [228]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve,
)

---

### Loading dataset

In [229]:
df = pd.read_csv("../data/data_1.csv")

In [230]:
df["event_time"] = pd.to_datetime(df["event_time"], format="ISO8601")
df = df.sort_values(["account_id", "event_time"]).reset_index(drop=True)

In [231]:
print(df.shape)

(206064, 6)


In [232]:
df.head()

,account_id,amount,event_time,is_home,is_domestic,is_fraud
0,ML_00000,10587000,2026-06-24 17:11:26.412047+00:00,True,True,False
1,ML_00000,9913000,2026-06-24 18:58:47.718187+00:00,True,True,False
2,ML_00000,11971000,2026-06-25 11:25:33.984989+00:00,True,True,False
3,ML_00002,7221000,2026-06-24 15:34:29.895982+00:00,True,True,False
4,ML_00002,5852000,2026-06-24 19:10:05.348565+00:00,True,True,False


---

### Feature Engineering

In [233]:
results = []
for _, grp in df.groupby("account_id"):
    grp = grp.sort_values("event_time").set_index("event_time")
    grp["avg_amount_24h"]         = grp["amount"].rolling("24h", closed="left", min_periods=1).mean()
    grp["tx_count_1h"]            = grp["amount"].rolling("1h",  closed="left", min_periods=0).count()
    grp["tx_count_3h"]            = grp["amount"].rolling("3h",  closed="left", min_periods=0).count()
    grp["time_since_last_tx_sec"] = grp.index.to_series().diff().dt.total_seconds()
    results.append(grp.reset_index())

df = pd.concat(results, ignore_index=True)

df["hour_of_day"]  = df["event_time"].dt.hour
df["amount_ratio"] = df["amount"] / df["avg_amount_24h"]

In [234]:
df[["account_id", "amount", "avg_amount_24h", "amount_ratio",
    "tx_count_1h", "tx_count_3h", "time_since_last_tx_sec", "hour_of_day", "is_fraud"]].head(10)

,account_id,amount,avg_amount_24h,amount_ratio,tx_count_1h,tx_count_3h,time_since_last_tx_sec,hour_of_day,is_fraud
0,ML_00000,10587000,NaN,NaN,0.0,0.0,NaN,17,False
1,ML_00000,9913000,1.058700e+07,0.936337,0.0,1.0,6441.306140,18,False
2,ML_00000,11971000,1.025000e+07,1.167902,0.0,0.0,59206.266802,11,False
3,ML_00002,7221000,NaN,NaN,0.0,0.0,NaN,15,False
4,ML_00002,5852000,7.221000e+06,0.810414,0.0,0.0,12935.452583,19,False
5,ML_00002,6221000,6.536500e+06,0.951733,1.0,1.0,2932.074202,19,False
6,ML_00003,12581000,NaN,NaN,0.0,0.0,NaN,12,False
7,ML_00003,11824000,1.258100e+07,0.939830,0.0,1.0,6912.625090,14,False
8,ML_00003,10678000,1.220250e+07,0.875067,1.0,2.0,59.859387,14,False
9,ML_00003,8719000,1.169433e+07,0.745575,2.0,3.0,114.925175,14,False


In [235]:
FEATURES = [
    "amount", "hour_of_day", "is_home", "is_domestic",
    "avg_amount_24h", "amount_ratio", "tx_count_1h", "tx_count_3h",  
    "time_since_last_tx_sec",
]

---

### Cleaning data

In [236]:
df["avg_amount_24h"]         = df["avg_amount_24h"].fillna(df["amount"])
df["amount_ratio"]           = df["amount_ratio"].fillna(1.0)
df["time_since_last_tx_sec"] = df["time_since_last_tx_sec"].fillna(0.0)

In [237]:
df["is_home"]     = df["is_home"].astype(int)
df["is_domestic"] = df["is_domestic"].astype(int)

In [238]:
df[["account_id", "amount", "avg_amount_24h", "amount_ratio",
    "tx_count_1h", "tx_count_3h", "time_since_last_tx_sec", "hour_of_day", "is_fraud"]].head(10)

,account_id,amount,avg_amount_24h,amount_ratio,tx_count_1h,tx_count_3h,time_since_last_tx_sec,hour_of_day,is_fraud
0,ML_00000,10587000,1.058700e+07,1.000000,0.0,0.0,0.000000,17,False
1,ML_00000,9913000,1.058700e+07,0.936337,0.0,1.0,6441.306140,18,False
2,ML_00000,11971000,1.025000e+07,1.167902,0.0,0.0,59206.266802,11,False
3,ML_00002,7221000,7.221000e+06,1.000000,0.0,0.0,0.000000,15,False
4,ML_00002,5852000,7.221000e+06,0.810414,0.0,0.0,12935.452583,19,False
5,ML_00002,6221000,6.536500e+06,0.951733,1.0,1.0,2932.074202,19,False
6,ML_00003,12581000,1.258100e+07,1.000000,0.0,0.0,0.000000,12,False
7,ML_00003,11824000,1.258100e+07,0.939830,0.0,1.0,6912.625090,14,False
8,ML_00003,10678000,1.220250e+07,0.875067,1.0,2.0,59.859387,14,False
9,ML_00003,8719000,1.169433e+07,0.745575,2.0,3.0,114.925175,14,False


---

### Spliting train/ test dataset

In [239]:
fraud_accounts  = df[df["is_fraud"]]["account_id"].unique()
normal_accounts = df[~df["is_fraud"]]["account_id"].unique()

In [240]:
rng = np.random.default_rng(42)
fraud_accounts  = np.array(fraud_accounts)
normal_accounts = np.array(normal_accounts)
rng.shuffle(fraud_accounts)
rng.shuffle(normal_accounts)

In [241]:
n_fraud_train  = int(len(fraud_accounts)  * 0.8)
n_normal_train = int(len(normal_accounts) * 0.8)

In [242]:
train_accounts = set(np.concatenate([fraud_accounts[:n_fraud_train],
                                     normal_accounts[:n_normal_train]]))
test_accounts  = set(np.concatenate([fraud_accounts[n_fraud_train:],
                                     normal_accounts[n_normal_train:]]))

In [243]:
FEATURES = [
    "amount", "hour_of_day", "is_home", "is_domestic",
    "avg_amount_24h", "amount_ratio", "tx_count_1h", "tx_count_3h",  
    "time_since_last_tx_sec",
]

In [244]:
train = df[df["account_id"].isin(train_accounts)]
test  = df[df["account_id"].isin(test_accounts)]

X_train, y_train = train[FEATURES], train["is_fraud"].astype(int)
X_test,  y_test  = test[FEATURES],  test["is_fraud"].astype(int)

In [245]:
print(f"Train: {len(train):,} rows  |  fraud: {y_train.sum():,} ({y_train.mean()*100:.2f}%)")
print(f"Test : {len(test):,} rows   |  fraud: {y_test.sum():,} ({y_test.mean()*100:.2f}%)")

Train: 166,040 rows  |  fraud: 4,083 (2.46%)
Test : 42,635 rows   |  fraud: 1,459 (3.42%)


---

### Training model

In [246]:
scale_pos_weight = int((y_train == 0).sum() / (y_train == 1).sum())

In [247]:
def objective(trial):
    params = {
        "n_estimators"         : 300,
        "max_depth"            : trial.suggest_int("max_depth", 3, 5),  
        "learning_rate"        : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"            : trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree"     : trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight"     : trial.suggest_int("min_child_weight", 3, 15), 
        "scale_pos_weight"     : scale_pos_weight,
        "eval_metric"          : "aucpr",
        "early_stopping_rounds": 20,
        "random_state"         : 42,
        "reg_alpha"            : trial.suggest_float("reg_alpha", 0.1, 2.0),   # L1
        "reg_lambda"           : trial.suggest_float("reg_lambda", 1.0, 5.0),  # L2
    }
    m = XGBClassifier(**params)
    m.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_prob = m.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, y_prob)

In [248]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, show_progress_bar=True)

Best trial: 32. Best value: 0.998931: 100%|██████████| 50/50 [00:35<00:00,  1.39it/s]


In [249]:
best = study.best_params
best.update({
    "n_estimators"         : 1000,
    "scale_pos_weight"     : scale_pos_weight,
    "eval_metric"          : "aucpr",
    "early_stopping_rounds": 20,
    "random_state"         : 42,
})

model = XGBClassifier(**best)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50,
)
print(f"Best iteration : {model.best_iteration}")

[0]	validation_0-aucpr:0.91888	validation_1-aucpr:0.92435
[50]	validation_0-aucpr:0.99899	validation_1-aucpr:0.99844
[100]	validation_0-aucpr:0.99958	validation_1-aucpr:0.99880
[150]	validation_0-aucpr:0.99975	validation_1-aucpr:0.99890
[186]	validation_0-aucpr:0.99982	validation_1-aucpr:0.99891
Best iteration : 166


---

### Evaluate

In [250]:
y_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

auc_roc = roc_auc_score(y_test, y_prob)
auc_pr  = average_precision_score(y_test, y_prob)

# Chọn threshold tối đa hóa F1
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
best_idx = f1s.argmax()
best_threshold = float(thresholds[best_idx])

y_pred = (y_prob >= best_threshold).astype(int)
p  = precision_score(y_test, y_pred)
r  = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Threshold : {best_threshold:.4f}\n")
print(f"Precision : {(p):.2f}  {'OK' if p > 0.80 else 'FAIL'}  (target > 0.80)")
print(f"Recall    : {(r):.2f}  {'OK' if r > 0.75 else 'FAIL'}  (target > 0.75)")
print(f"F1-Score  : {(f1):.2f}  {'OK' if f1 > 0.78 else 'FAIL'}  (target > 0.78)")
print(f"AUC-ROC   : {(auc_roc):.2f}  {'OK' if auc_roc > 0.85 else 'FAIL'}  (target > 0.85)")
print(f"AUC-PR    : {(auc_pr):.2f}  {'OK' if auc_pr > 0.70 else 'FAIL'}  (target > 0.70)")


Threshold : 0.9478

Precision : 0.91  OK  (target > 0.80)
Recall    : 0.94  OK  (target > 0.75)
F1-Score  : 0.90  OK  (target > 0.78)
AUC-ROC   : 0.93  OK  (target > 0.85)
AUC-PR    : 0.91  OK  (target > 0.70)


---

### Save Model

In [252]:
out_path = Path("../data/model.json")
out_path.parent.mkdir(exist_ok=True)

model.get_booster().save_model(str(out_path))

print(f"Saved : {out_path}")
print(f"Size  : {out_path.stat().st_size / 1024:.1f} KB")
print(f"Threshold dùng trong Flink: {best_threshold:.4f}")

Saved : ../data/model.json
Size  : 411.3 KB
Threshold dùng trong Flink: 0.9478
